# Projeto 2 Fork: METABRIC Clinico

Notebook narrativo do fork clinico para melhoria continua do Projeto 2.

## Objetivo

Prever `Overall Survival Status = Deceased` sem usar `Overall Survival (Months)` como preditor principal. O tempo de sobrevida fica reservado para analise de sensibilidade/follow-up.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path('projeto_2_neuro_fuzzy_metabric_clinico')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / 'src'))

from breast_cancer_survival.data import load_raw_data, sanitize_data, build_dataset_contract
from breast_cancer_survival.features import add_clinical_features, split_features_target

raw = load_raw_data()
clean = sanitize_data(raw)
engineered = add_clinical_features(clean)
build_dataset_contract(raw, clean)


## Metadados e dicionario

Os arquivos abaixo sao gerados pelos scripts `01_validate_data.py` e `02_eda.py`.

In [ ]:
tables = PROJECT_DIR / 'reports' / 'tables'
pd.read_csv(tables / 'metadata_profile.csv').head(12)


## Politica de features

O modelo principal remove identificador, meses de sobrevida e variaveis pos-acompanhamento.

In [ ]:
X, y = split_features_target(engineered, include_survival_months=False)
print(X.shape, y.value_counts().sort_index().to_dict())
sorted(set(['patient_id', 'survival_months', 'relapse_free_status']) & set(X.columns))


## Resultados sem vazamento


In [ ]:
pd.read_csv(tables / 'model_comparison_no_leakage.csv')[['model','accuracy','precision','recall','f2','pr_auc','false_negatives']]


## Ensemble e cenarios de threshold


In [ ]:
pd.read_csv(tables / 'ensemble_summary.csv')[['threshold','accuracy','precision','recall','f2','pr_auc','false_negatives','false_positives']]


## Neuro-fuzzy cooperativo


In [ ]:
pd.read_csv(tables / 'neuro_fuzzy_comparison.csv')[['model','accuracy','precision','recall','f2','pr_auc','false_negatives']]


## Conclusao

O METABRIC clinico e mais adequado que o SEER atual para aprendizado continuo: possui tratamentos, biomarcadores e subtipo molecular, melhora PR AUC/F2 e permite discutir trade-offs de threshold com mais sinal clinico.